In [4]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'
import torch
from sentence_transformers import SentenceTransformer
print('Đã load Torch thành công!')


Đã load Torch thành công!


In [ ]:
import os
import json
import re
import pandas as pd
import numpy as np
import sys
import subprocess
import gc
from dotenv import load_dotenv 
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
project_root = Path.cwd().parent
src_path = project_root / "Seg_OCR_Tri" / "src"
sys.path.insert(0, str(src_path))
load_dotenv()
os.environ["HF_TOKEN"]= str(os.getenv("HF_TOKEN"))
def build_xgboost_dataset_from_sroie(img_folder, key_folder, output_csv="xgboost_dataset.csv"):
    
    features_list = []
    
    img_files = [f for f in os.listdir(img_folder) if f.endswith(('.jpg', '.png'))]
    print(f"🚀 Bắt đầu quét {len(img_files)} hóa đơn từ tập SROIE...")
    
    # --- OPTIMIZATION: LOAD MODEL ONCE ---
    embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    POSITIVE_VECTOR = embedding_model.encode(["order total, total amount to pay, grand total, final total"])[0]
    NEGATIVE_VECTOR = embedding_model.encode(["transaction ID, reference code, authorization, phone number, date, time, credit card, subtotal, tax amount, change due, item code, sku, barcode"])[0]
    CURRENCY_PATTERN = re.compile(r'[\$€£¥₩₹₽₺₴₦฿₫₱¢]|USD|EUR|VND|JPY|GBP|AUD|CAD|SGD', re.IGNORECASE)
    
    def get_candidates_list_only(easyocr_results, img_height):
        
        candidates = []
        all_values = []
        metadata_blacklist = ['id', 'code', 'trn', 'auth', 'seq', 'acq', 'phone', 'tel', 'date', 'time', 'mastercard', 'visa']
        
        for idx, res in enumerate(easyocr_results):
            bbox, text = res
            text_clean = text.strip()
            
            for match in re.finditer(r'\b\d+(?:[\.,:]\d+)*\b', text_clean):
                num_str = match.group()
                start_idx, end_idx = match.span()
                
                raw_digits = re.sub(r'\D', '', num_str)
                if not raw_digits or len(raw_digits) > 12:
                    continue
                    
                s = num_str.replace(" ", "")
                if '.' in s or ',' in s or ':' in s:
                    sep = '.' if '.' in s else (',' if ',' in s else ':')
                    parts = s.split(sep)
                    last_part = parts[-1]
                    
                    if len(last_part) == 2:
                        main_part = "".join(parts[:-1]).replace('.', '').replace(',', '')
                        clean_val = float(f"{main_part}.{last_part}")
                    else:
                        clean_val = float(s.replace('.', '').replace(',', '').replace(':', ''))
                else:
                    clean_val = float(s)
                    
                all_values.append(clean_val)
                y_center = np.mean([point[1] for point in bbox])
                normalized_y = y_center / img_height
                
                left_window = text_clean[max(0, start_idx - 25):start_idx]
                right_window = text_clean[end_idx:min(len(text_clean), end_idx + 15)]
                
                has_letters = bool(re.search(r'[a-zA-Z]', left_window))
                has_numbers = bool(re.search(r'\d', left_window))
                
                if not has_letters and not has_numbers and idx > 0:
                    potential_prev_line = easyocr_results[idx - 1][1]
                    if not re.search(r'\d', potential_prev_line):
                        prev_line = potential_prev_line[-25:]
                    else:
                        prev_line = ""
                else:
                    prev_line = ""
                    
                neighbor_text = f"{prev_line} {left_window} {right_window}".strip()
                    
                candidates.append({
                    "value": clean_val,
                    "normalized_y": normalized_y,
                    "neighbor_text": neighbor_text
                })
                
        if not candidates:
            return []
            
        max_bill_value = max(all_values) if all_values else 1
        
        for cand in candidates:
            is_max = 1.0 if cand["value"] == max_bill_value else 0.0
            has_currency = 1.0 if CURRENCY_PATTERN.search(cand["neighbor_text"]) else 0.0
            
            if cand["neighbor_text"]:
                if not re.search(r'[a-zA-Z]', cand["neighbor_text"]):
                    semantic_sim = 0.0
                else:
                    cand_vector = embedding_model.encode([cand["neighbor_text"]])[0]
                    pos_sim = cosine_similarity([cand_vector],[POSITIVE_VECTOR])[0][0]
                    neg_sim = cosine_similarity([cand_vector],[NEGATIVE_VECTOR])[0][0]
                    
                    if pos_sim < 0.25 or neg_sim > pos_sim:
                        semantic_sim = 0.0
                    else:
                        semantic_sim = pos_sim
                        
                    if any(word in cand["neighbor_text"].lower() for word in metadata_blacklist):
                        semantic_sim *= 0.1
            else:
                semantic_sim = 0.0
                
            cand["semantic_sim"] = semantic_sim
            cand["is_max"] = is_max
            cand["has_currency"] = has_currency
            cand["text_length"] = len(cand["neighbor_text"])
            
        return candidates
    
    ocr_results_file = "temp_ocr_results.json"
    print("🚀 Đang chạy PaddleOCR ngầm để tránh đụng độ bộ nhớ với PyTorch...")
    subprocess.run([sys.executable, "ocr_worker.py", img_folder, ocr_results_file], check=True)
    
    with open(ocr_results_file, 'r', encoding='utf-8') as f:
        all_ocr_results = json.load(f)
        
    for count, img_name in enumerate(img_files, 1):
        try:
            # 1. Khớp tên file ảnh với file đáp án (Đã sửa thành .json)
            base_name = os.path.splitext(img_name)[0]
            img_path = os.path.join(img_folder, img_name)
            key_path = os.path.join(key_folder, f"{base_name}.json")
            
            if not os.path.exists(key_path):
                print(f"❌ Bỏ qua {img_name}: KHÔNG TÌM THẤY file {base_name}.json")
                continue
                
            # 2. Đọc trực tiếp trường "total" bằng thư viện JSON
            with open(key_path, 'r', encoding='utf-8') as f:
                try:
                    key_data = json.load(f)
                except json.JSONDecodeError:
                    print(f"⚠️ Bỏ qua {img_name}: File JSON bị lỗi định dạng")
                    continue
                
                # Kiểm tra xem trường "total" có tồn tại không
                if "total" not in key_data or key_data["total"] is None:
                    print(f"⚠️ Bỏ qua {img_name}: Không có trường 'total'")
                    continue
                
                raw_total_str= str(key_data["total"])
                clean_total_str = re.sub(r'[^\d\.,]', '', raw_total_str)
                clean_total_str = clean_total_str.replace(',', '.')
                if not clean_total_str:
                    print(f"⚠️ Bỏ qua {img_name}: Không tìm thấy số trong đáp án '{raw_total_str}'")
                    continue
                    
                ground_truth_total = float(clean_total_str)
                
            # 3. Lấy kết quả OCR từ file JSON (đã chạy ngầm xong)
            if img_name not in all_ocr_results:
                continue
            ocr_data = all_ocr_results[img_name]["ocr_data"]
            img_height = all_ocr_results[img_name]["img_height"]
            
            candidates = get_candidates_list_only(ocr_data, img_height) 
            
            if not candidates:
                continue
                
            max_val = max([c["value"] for c in candidates])
            
            # 4. TỰ ĐỘNG DÁN NHÃN (AUTO-LABELING)
            for cand in candidates:
                has_currency = 1.0 if re.search(r'[\$€£¥₩₹₽₺₴₦฿₫₱¢]|USD|EUR|VND', cand["neighbor_text"], re.IGNORECASE) else 0.0
                
                label = 1 if abs(cand["value"] - ground_truth_total) < 0.01 else 0
                
                features_list.append({
                    "semantic_sim": cand.get("semantic_sim", 0.0), 
                    "normalized_y": cand["normalized_y"],
                    "is_max": 1.0 if cand["value"] == max_val else 0.0,
                    "text_length": len(cand["neighbor_text"]),
                    "has_currency": has_currency,
                    "label": label
                })
            print(f"[{count}/{len(img_files)}] Xử lý xong: {img_name} | Total chuẩn: {ground_truth_total}")
            gc.collect()
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"⚠️ Bỏ qua {img_name} do lỗi: {e}")
            continue

    # 5. Lưu toàn bộ thành file CSV
    df = pd.DataFrame(features_list)
    df.to_csv(output_csv, index=False)
    print(f"✅ THÀNH CÔNG RỰC RỠ! Đã xuất xưởng dataset tại: {output_csv} với {len(df)} dòng đặc trưng.")
build_xgboost_dataset_from_sroie(
    img_folder=r"D:\VGU\Intro to AI\invoice-reader\Candidate_classification\img",
    key_folder=r"D:\VGU\Intro to AI\invoice-reader\Candidate_classification\key"
)



🚀 Bắt đầu quét 626 hóa đơn từ tập SROIE...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12358.29it/s]


🚀 Đang chạy PaddleOCR ngầm để tránh đụng độ bộ nhớ với PyTorch...
⚠️ Bỏ qua 000.jpg do lỗi: could not convert string to float: '5:55.57'
[2/626] Xử lý xong: 001.jpg | Total chuẩn: 60.3
[3/626] Xử lý xong: 002.jpg | Total chuẩn: 33.9
[4/626] Xử lý xong: 003.jpg | Total chuẩn: 80.9
[5/626] Xử lý xong: 004.jpg | Total chuẩn: 30.9
[6/626] Xử lý xong: 005.jpg | Total chuẩn: 31.0
[7/626] Xử lý xong: 006.jpg | Total chuẩn: 327.0
[8/626] Xử lý xong: 007.jpg | Total chuẩn: 20.0
[9/626] Xử lý xong: 008.jpg | Total chuẩn: 112.45
[10/626] Xử lý xong: 009.jpg | Total chuẩn: 26.6
[11/626] Xử lý xong: 010.jpg | Total chuẩn: 14.1
[12/626] Xử lý xong: 011.jpg | Total chuẩn: 15.0
[13/626] Xử lý xong: 012.jpg | Total chuẩn: 15.9
[14/626] Xử lý xong: 013.jpg | Total chuẩn: 15.0
[15/626] Xử lý xong: 014.jpg | Total chuẩn: 32.7
[16/626] Xử lý xong: 015.jpg | Total chuẩn: 15.9
[17/626] Xử lý xong: 016.jpg | Total chuẩn: 73.0
[18/626] Xử lý xong: 017.jpg | Total chuẩn: 39.8
[19/626] Xử lý xong: 018.jpg | Tota